Jawabannya adalah: **Prediksi harus dilakukan menggunakan data dari Supabase (`store_sales`), lalu hasilnya di-update ke tabel `inventory` di Supabase.**

Berikut adalah penjelasan logika dan alur integrasi antara skema database Supabase Anda dengan model prediksi FastAPI agar sistem Anda terintegrasi secara utuh.

---

### 1. Bagaimana Data Saling Terhubung?

Jika melihat skema Supabase Anda, terdapat dua alur utama:

```mermaid
graph TD
    subgraph Supabase Database
        SS[store_sales] -->|Historis Penjualan| API
        I[inventory] <---|Update Hasil Prediksi| API
    end

    subgraph FastAPI (AI Engine)
        API[POST /forecast/run] -->|Hitung ML Model| ML[XGBoost / Prophet / LSTM]
        ML -->|Output Prediksi & AI Insight| API
    end
```

1. **Input (Data Historis untuk Training):**
   Model FastAPI Anda membutuhkan data historis penjualan untuk belajar (training). Data ini diambil dari tabel **`store_sales`** di Supabase (kolom `date`, `warehouse` [sebagai store], `item`, dan `sales`).
2. **Output (Hasil Prediksi untuk Manajemen Inventori):**
   Setelah model FastAPI mengeluarkan angka prediksi, metrik, dan rekomendasi AI, Anda melakukan **`UPDATE`** ke tabel **`inventory`** di Supabase pada kolom-kolom yang sudah Anda siapkan (`predicted_demand`, `shortage`, `status`, dan `recommended_action`).

---

### 2. Pemetaan Kolom dari API ke Tabel `inventory` Supabase

Setelah Anda menembak `/forecast/run` dari frontend, Anda akan menerima response JSON. Anda harus memetakan response tersebut untuk meng-update tabel `inventory` sebagai berikut:

| Kolom di Tabel `inventory` | Diambil / Dihitung dari Response API | Logika / Contoh Pengisian |
| :--- | :--- | :--- |
| **`predicted_demand`** | Hasil penjumlahan `predicted_sales` di dalam array `forecast` (misal untuk 30 atau 90 hari ke depan). | Jika total prediksi sales 90 hari ke depan adalah 400 unit, maka diisi **`400`**. |
| **`shortage`** | Diperoleh dari rumus: `predicted_demand` - `current_stock`. | Jika prediksi demand = 400, sedangkan `current_stock` saat ini = 250, maka shortage = **`150`** unit. (Jika hasilnya minus, isi `0` / tidak ada shortage). |
| **`status`** | Diambil dari `insight.stockout_risk` atau kondisi shortage. | `"High Risk"`, `"Medium Risk"`, atau `"Low Risk"`. |
| **`recommended_action`** | Diambil dari `insight.recommended_action` (atau bullet point dari `/forecast/ai-insight`). | `"Increase safety stock to 50 units for Jan W2."` |

---

### 3. Alur Eksekusi (Workflow Integrasi Frontend & Backend)

Berikut adalah skenario bagaimana frontend Anda bekerja menggunakan Supabase + FastAPI:

#### Skenario: Tombol "Run Sync Forecast" di halaman Inventory Management
1. **Frontend** mengambil daftar produk aktif dari tabel `products` dan `inventory` di Supabase.
2. Untuk setiap produk (atau produk yang dipilih user), **Frontend** memicu request ke FastAPI:
   ```http
   POST http://127.0.0.1:8000/forecast/run
   ```
   *(Catatan: Backend FastAPI akan meng-query tabel `store_sales` di Supabase untuk mendapatkan data historis ter-update sebagai input training model).*
3. **FastAPI** mengembalikan output prediksi.
4. **Frontend** (atau middleware backend Anda) menerima response tersebut, lalu langsung mengirim perintah **`UPDATE` ke Supabase** untuk memperbarui baris inventory barang tersebut:
   ```javascript
   // Contoh implementasi di Frontend menggunakan Supabase JS Client:
   const totalPredictedDemand = data.forecast.reduce((sum, item) => sum + item.predicted_sales, 0);
   const currentStock = 250; // Diambil dari inventory saat ini
   const shortage = Math.max(0, totalPredictedDemand - currentStock);

   const { error } = await supabase
     .from('inventory')
     .update({
       predicted_demand: Math.round(totalPredictedDemand),
       shortage: Math.round(shortage),
       status: data.insight.stockout_risk + " Risk",
       recommended_action: data.insight.recommended_action
     })
     .eq('product_id', productId)
     .eq('warehouse_id', warehouseId);
   ```
5. Tabel `inventory` kini terisi dengan data analisis terbaru, dan halaman **Inventory Management** di frontend Anda akan langsung menampilkan status ter-update secara real-time!

> **Kesimpulan:** 
> Anda melakukan **prediksi menggunakan data historis dari tabel `store_sales` Supabase**, lalu **menyimpan/menuliskan hasil kalkulasi prediksi tersebut kembali ke tabel `inventory` Supabase** agar sistem sinkron dan data inventori Anda selalu relevan dengan proyeksi masa depan.
